In this notebook, we look at a workflow for extracting a single DFT frame from an audio file.

Let's load the audio file first:

In [1]:
from audiospylt.audio_utils import load_audio_sample_and_preview

# Set options here, then make a single call that applies them.
#
# Parameters (audio_opts):
# - wav_source: local path or URL to an audio file
# - desired_sample_rate: None keeps the native rate, or set an int (e.g., 44100)
# - convert_to_mono: True mixes/downmixes to mono on load
# - download_mode: for URL sources, where to store the downloaded file: 'cwd' | 'dir' | 'temp'
#   - 'cwd' saves the downloaded file into the current working directory (backward-compatibility default)
#   - 'temp' downloads to a temporary file and deletes it after decoding (no leftover file)
# - download_dir: directory used when download_mode='dir'
# - show_load_bar: show a notebook-friendly tqdm download progress bar (URL only)
# - show_properties: print channels, sample rate, and source
# - show_waveform: plot the time-domain waveform (uses the playback channel)
# - plot_width: optional width for the waveform plot
# - plot_height: optional height for the waveform plot
# - plot_title_source: which string to show in the title: 'filename' | 'source'
# - plot_title_mode: how to show it: 'full' | 'basename'
# - play_audio: play audio in the notebook via IPython Audio
# - playback_fs_target_name: export/playback sample-rate preset for render_audio
#   - use 'source' to keep the loaded SR (recommended when desired_sample_rate=None)
#   - or choose a preset: '44.1kHz', '48kHz', '88.2kHz', '96kHz', '192kHz'
# - playback_bit_rate: export bit depth for render_audio (16 or 24)
# - playback_save_audio: True writes a WAV into ./rendered_audio; False keeps it in-memory
# - playback_sanitize: replace NaN/Inf with finite values before export/playback
# - source_verbose: verbosity for loading/downloading the source
# - playback_verbose: verbosity for render_audio

audio_opts = dict(
    wav_source='https://ccrma.stanford.edu/~jos/mp3/viola.mp3',
    desired_sample_rate=None,
    convert_to_mono=True,
    download_mode='temp',
    download_dir=None,
    show_load_bar=True,
    show_properties=True,
    show_waveform=True,
    play_audio=True,
    playback_fs_target_name='source',
    playback_bit_rate=24,
    playback_save_audio=False,
    playback_sanitize=True,
    source_verbose=True,
    playback_verbose=False,
    plot_width=1200,
    plot_height=600,
    plot_title_source='source',
    plot_title_mode='full',
)

audio_data, sample_rate, audio_info = load_audio_sample_and_preview(**audio_opts)


Audio downloaded from URL into a temporary file: /tmp/tmptrxbjre0.mp3
Number of audio channels: 1
Sampling rate: 44100 Hz
WAV file loaded from https://ccrma.stanford.edu/~jos/mp3/viola.mp3
Total duration: 6.792 seconds


Let's locate the exact position of our point of interest and apply fade-ins and fade-outs if needed.

In [2]:
from audiospylt.audio_utils import trim_and_fade_and_render

# Set options here, then make a single call that applies them.
#
# Parameters (cut_opts):
# - start_time/end_time: trim window in seconds
# - add_fades: apply fade-in/fade-out envelopes
# - fade_in_duration/fade_out_duration: fade lengths in seconds
# - fade_in_exponent/fade_out_exponent: shape exponents (>0); smaller = more linear, larger = more curved
# - plot_width: optional width for the trim/fade plot (pixels)
# - plot_height: optional height for the trim/fade plot (pixels)
# - plot_title_source: which string to show in the title: 'filename' | 'source'
# - plot_title_mode: how to show it: 'full' | 'basename'
# - cut_sanitize: sanitize NaN/Inf in the returned cut_audio_data (keeps downstream analysis stable)
# - cut_verbose: print a warning if sanitization happens
# - render: whether to call render_audio after trimming
# - fs_target_name: 'source' keeps the current SR; or choose a preset like '44.1kHz'
# - bit_rate: 16 or 24
# - save_audio: True writes a WAV into ./rendered_audio; False keeps it in-memory
# - player: play audio widget in the notebook
# - sanitize/verbose: forwarded to render_audio

cut_opts = dict(
    start_time=4.37,
    end_time=4.75,
    add_fades=True,
    fade_in_duration=0.03,
    fade_out_duration=0.05,
    fade_in_exponent=0.8,
    fade_out_exponent=1.5,
    plot_width=1000,
    plot_height=None,
    plot_title_source='source',
    plot_title_mode='full',
    cut_sanitize=True,
    cut_verbose=True,
    render=True,
    fs_target_name='source',
    bit_rate=24,
    save_audio=False,
    player=True,
    sanitize=True,
    verbose=True,
)

cut_audio_data, cut_duration, cut_render_result = trim_and_fade_and_render(
    audio_data=audio_data,
    sample_rate=sample_rate,
    audio_info=audio_info,
    **cut_opts,
)


We can also inspect the corresponding spectral analysis of the selected point of interest:

In [3]:
from audiospylt.audio_utils import plot_spectrogram_with_waveform

# Set options here, then make a single call that applies them.
#
# Parameters (spec_opts):
# - y_axis_mode: 'linear' | 'log' | 'mel' | 'mixed'
# - y_axis_mix: 0..1 (only for 'mixed')
# - mixed_log_floor_hz: >0 (only for 'mixed')
# - amp_scale: 'db' (log) | 'linear' (raw) | 'linear_norm' (0-1)
# - power_law_gamma: <1.0 (e.g. 0.5) to compress the range for linear modes
# - n_fft: FFT window length in samples
# - window_type: scipy window name (e.g., 'hann'); full list: https://docs.scipy.org/doc/scipy/reference/signal.windows.html
# - overlap: 0..(<0.95) fraction of window overlap
# - oversample_factor: >=1.0 zero-padding multiplier (nfft = n_fft * factor)
# - boundary:
#   - 'zeros' (centered): pads both sides; the first frame includes pre-zero padding
#   - 'zeros_end' (start-aligned): pads only the end; the first full window starts at t=0
# - padded: if True, pads the end so the last frame reaches the end
# - time_reference:
#   - 'center': x is window centers (scipy default)
#   - 'start': x is window start times (removes empty area at the beginning for 'zeros_end')
# - time_range: 'signal' shows 0..duration; 'stft' shows only the returned STFT span
# - waveform_mode:
#   - 'integrated': waveform subplot under the spectrogram (single figure)
#   - 'separate': waveform shown as a separate figure
#   - 'none': spectrogram only
# - waveform_height_ratio: height share for waveform when integrated
# - waveform_line_width: waveform line width
# - waveform_max_points: downsample waveform for speed (int max points), or 'sample_rate' to use sample_rate
# - mel_bins/mel_fmax: only used when y_axis_mode='mel'
# - scaling: 'density' | 'spectrum'
# - mode: 'magnitude' | 'psd'
# - cmap: Plotly colorscale name
# - show: display the figure(s)
# - print_info: print derived STFT settings under the plot (hop, overlap %, redundancy, etc.)
# - plot_width/plot_height: optional figure size (pixels)
# - plot_title_source: which string to show in the title: 'filename' | 'source' (uses audio_info)
# - plot_title_mode: how to show it: 'full' | 'basename'

spec_opts = dict(
    audio_info=audio_info,

    # Axis scaling:
    y_axis_mode="mel",
    y_axis_mix=0.5,
    mixed_log_floor_hz=1.0,
    amp_scale="db",
    power_law_gamma=0.1,

    # STFT params:
    n_fft=2048,
    window_type="hann",
    overlap=0.5,
    oversample_factor=2.0,

    # Edge framing / time base:
    boundary="zeros_end",
    padded=True,
    time_reference="start",
    time_range="signal",

    # Optional synchronous waveform:
    waveform_mode="integrated",
    waveform_height_ratio=0.25,
    waveform_line_width=1.0,
    waveform_max_points="sample_rate",

    # Mel-only params:
    mel_bins=128,
    mel_fmax=sample_rate / 2,

    # Scaling/appearance:
    scaling="density",
    mode="magnitude",
    cmap="Magma",
    show=True,
    print_info=True,

    # Plot size / title naming:
    plot_width=1000,
    plot_height=None,
    plot_title_source="source",
    plot_title_mode="full",
)

fig, spec_info = plot_spectrogram_with_waveform(cut_audio_data, sample_rate, **spec_opts)



Spectrogram settings:
- sample_rate: 44100.0 Hz
- n_fft: 2048  (window: 0.0464s)
- window_type: hann
- hop_length: 1024  (hop: 0.0232s)
- overlap: 0.500 (50.0%)
- redundancy (n_fft / hop): 2.000x
- oversample_factor: 2.0  (bin width: 10.77 Hz)
- boundary: zeros_end; padded: True
- time_reference: start; time_range: signal
- amplitude: db (gamma=0.1)
- frames: 17  (padded at end: ~1675 samples, 0.0380s)


Now we can use a threshold-based function to filter out partials. We can also use additional window functions if needed; the full list can be found here: https://docs.scipy.org/doc/scipy/reference/signal.windows.html

In [4]:
from audiospylt.dft_analysis import analyze_signal

# Peak-picking parameters (keep within sensible bounds):
# - window_type: scipy window name
# - thresh_amp_low/high: peak height bounds in *linear FFT magnitude* units (0 < low < high)
# - thresh_freq_low/high: keep peaks within [low, high] Hz (high <= Nyquist or None)
# - prominence: (advanced) raw scipy prominence in the same units as the spectrum amplitude
# - prominence_rel: (recommended) [0,1] prominence as a fraction of max(spec)
# - width: (advanced) raw scipy width in FFT bins
# - width_hz: (recommended) width in Hz (converted to bins)
# - distance_hz: (optional) minimum spacing between peaks in Hz (helps avoid picking adjacent bins)
# - freq_axis_mode: 'linear' | 'log' | 'mel' | 'mixed' (plot only)
# - freq_axis_mix: [0,1] only for 'mixed' (0 = linear spacing, 1 = log-like spacing)
# - mixed_log_floor_hz: >0 only for 'mixed' (plot only)
# - amp_axis_mode: 'linear' | 'log' | 'mixed' (plot only)
# - amp_axis_mix: [0,1] only for amp_axis_mode='mixed'
# - amp_log_floor: >0 floor used for log/mixed amplitude plotting (avoids log(0))
# - auto_plot_range: auto-zoom plot ranges based on freq/amp thresholds (+ padding)
# - freq_plot_pad_hz/freq_plot_pad_frac: padding around [thresh_freq_low, thresh_freq_high] (Hz or fraction of span)
# - amp_plot_pad/amp_plot_pad_frac: additive padding around [thresh_amp_low, thresh_amp_high] (linear/mixed)
# - amp_plot_pad_ratio: multiplicative padding for log amp axis
# - plot_width/plot_height: plot size (pixels) for waveform+window preview and spectrum plot
# - plot_title_source: which string to show in plot titles: 'filename' | 'source'
# - plot_title_mode: how to show it: 'full' | 'basename'
# - show_windowed_waveform: if True, show the waveform after applying window_type, with the window curve overlaid in red (shown first)
# - partial_tracking: if True, keep only peaks within +/- partial_bandwidth_hz of k*f0 (applies after initial peak filtering)
# - f0: fundamental frequency in Hz (required when partial_tracking=True)
# - partial_bandwidth_hz: bandwidth in Hz around each partial (k*f0)
# - plot_partials: if True, overlay partial center lines and +/- bandwidth bands on the spectrum
#
# Highlight / styling parameters (Plotly color strings):
# - spectrum_color: None (default) or a color (e.g., 'royalblue', '#1f77b4', 'rgba(31,119,180,1)')
# - peaks_color: None (default) or a color string
# - threshold_color: color for threshold lines (default: 'Red')
# - partials_color: color for partial center lines (default: 'rgba(0, 180, 0, 0.65)')
# - partials_band_fill: fill color for partial bands (default: 'rgba(0, 180, 0, 0.10)')
#
# Notes:
# - thresh_amp_* values are in *linear FFT magnitude* units (not dB).
# - For amp_axis_mode='mixed', y-axis tick labels are mapped back to raw amplitude units.

window_type = "hann"
thresh_amp_low = 0.0015
thresh_amp_high = 0.025
thresh_freq_low = 30
thresh_freq_high = 1250

# Prefer the intuitive variants:
prominence_rel = 0.005  # 0-1 range, try smaller values for more sensitive detection
width_hz = 3        # try bigger values to prefer broader peaks
distance_hz = 5     # try bigger values to avoid adjacent-bin peaks

# Peak detection parameters
# Keep these as None unless you want raw scipy behavior:
prominence = None
width = None

# Plot axis scaling:
freq_axis_mode = "linear"
freq_axis_mix = 0.5
mixed_log_floor_hz = 1.0

amp_axis_mode = "linear"
amp_axis_mix = 0.5
amp_log_floor = 1e-12

# Auto-zoom plot ranges to your thresholds (+ padding):
auto_plot_range = True
freq_plot_pad_hz = None      # set e.g. 50 for +/-50 Hz padding
freq_plot_pad_frac = 0.05    # used when freq_plot_pad_hz is None
amp_plot_pad = None          # set e.g. 0.01 for +/-0.01 amplitude padding
amp_plot_pad_frac = 0.10     # used when amp_plot_pad is None
amp_plot_pad_ratio = 0.15    # only used when amp_axis_mode='log'

# Plot size / naming:
plot_width = 800
plot_height = None
plot_title_source = "source"   # "filename" | "source"
plot_title_mode = "basename"     # "basename" | "full"
show_windowed_waveform = True

# Partial tracking (after initial peak filtering):
partial_tracking = True
f0 = 263.0
partial_bandwidth_hz = 10.0
plot_partials = True

# Highlight colors (optional; keep None to use Plotly defaults):
spectrum_color = None
peaks_color = None
threshold_color = "Red"
partials_color = "rgba(0, 180, 0, 0.65)"
partials_band_fill = "rgba(0, 180, 0, 0.20)"

show_peaks = True
show_plot = True

plot_title_name = (
    audio_info["wav_source"] if plot_title_source == "source" else audio_info["wav_filename"]
)

peaks_df = analyze_signal(
    signal=cut_audio_data,
    sr=sample_rate,
    filename=plot_title_name,
    window_type=window_type,
    thresh_amp_low=thresh_amp_low,
    thresh_amp_high=thresh_amp_high,
    thresh_freq_low=thresh_freq_low,
    thresh_freq_high=thresh_freq_high,
    prominence=prominence,
    width=width,
    prominence_rel=prominence_rel,
    width_hz=width_hz,
    distance_hz=distance_hz,
    freq_axis_mode=freq_axis_mode,
    freq_axis_mix=freq_axis_mix,
    mixed_log_floor_hz=mixed_log_floor_hz,
    amp_axis_mode=amp_axis_mode,
    amp_axis_mix=amp_axis_mix,
    amp_log_floor=amp_log_floor,
    auto_plot_range=auto_plot_range,
    freq_plot_pad_hz=freq_plot_pad_hz,
    freq_plot_pad_frac=freq_plot_pad_frac,
    amp_plot_pad=amp_plot_pad,
    amp_plot_pad_frac=amp_plot_pad_frac,
    amp_plot_pad_ratio=amp_plot_pad_ratio,
    show_peaks=show_peaks,
    show_plot=show_plot,
    plot_width=plot_width,
    plot_height=plot_height,
    plot_title_name=plot_title_name,
    plot_title_mode=plot_title_mode,
    show_windowed_waveform=show_windowed_waveform,
    partial_tracking=partial_tracking,
    f0=f0,
    partial_bandwidth_hz=partial_bandwidth_hz,
    plot_partials=plot_partials,
    spectrum_color=spectrum_color,
    peaks_color=peaks_color,
    threshold_color=threshold_color,
    partials_color=partials_color,
    partials_band_fill=partials_band_fill,
)

File name: https://ccrma.stanford.edu/~jos/mp3/viola.mp3
Duration (s): 0.38
Sampling rate (Hz): 44100

Maximum amplitude value: 0.020703
Total number of bands: 8380
Frequency resolution (Hz): 2.631579

Amplitude Threshold 1: 0.0015
Amplitude Threshold 2: 0.025
Frequency Threshold 1 (Hz): 30
Frequency Threshold 2 (Hz): 1250

Peaks:


,Frequency (Hz),Amplitude,Partial #,Delta from partial (Hz)
0,263.157895,0.020703,1,0.157895
1,526.315789,0.007845,2,0.315789
2,784.210526,0.014084,3,-4.789474
3,797.368421,0.006947,3,8.368421
4,1055.263158,0.020562,4,3.263158


Now we can save the extracted partials table:

In [6]:
from audiospylt.io_utils import save_df_tsv

# Save the amp/freq table (peaks_df) for reuse.
save_df_tsv(peaks_df, "../tsv/voice-single3.tsv")

Data saved successfully to /home/eggi/Nextcloud/code/public_repos/audiospylt/tutorials_tech\../tsv/voice-single3.tsv at 2026-02-24 16:38:53.623971.


'../tsv/voice-single3.tsv'